# Random Forest Benchmark — GPU (Colab)

Self-contained: all implementation is inlined — no imports from the project codebase.
Uses `XGBRFRegressor` with `device='cuda'` for GPU-accelerated random forest.

**Before running:**
1. Runtime → Change runtime type → T4 GPU (or A100)
2. Provide the repo and data (Setup cell)
3. Edit the **Constants** cell, then *Run All*

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "No GPU detected — switch runtime to GPU first.")

In [ ]:
%%capture
!pip install \
    "pandas==2.2.2" \
    "numpy==1.26.4" \
    "scikit-learn==1.5.1" \
    "xgboost==2.1.1" \
    "scipy==1.13.1" \
    "tqdm==4.66.4"
print("Installation complete.")

## Setup

Uncomment **one** option, run it, then continue.

In [ ]:
# Option A: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Option B: Clone from GitHub
# !git clone https://github.com/YOUR_USER/Volf.git /content/Volf

In [ ]:
from pathlib import Path

# ── EDIT THESE ────────────────────────────────────────────────────────────────
DATA_DIR       = Path("/content/Volf/data")   # directory that contains ag/ and benchmark/
CONFIG_WHEAT   = Path("/content/Volf/config/wheat/random_forest_mean.json")
CONFIG_CORN    = Path("/content/Volf/config/corn/random_forest_mean.json")
CONFIG_SOYBEAN = Path("/content/Volf/config/soybean/random_forest_mean.json")
# ──────────────────────────────────────────────────────────────────────────────

print(f"DATA_DIR : {DATA_DIR}")

## Imports

In [ ]:
import dataclasses
import datetime
import json
import logging
import math
import re
import time
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBRFRegressor

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s — %(message)s",
    force=True,
)

## Implementation

In [ ]:
# ── Feature-group column lists (from src/benchmark/utils.py) ──────────────────

CLIMATE_COLUMNS = [
    "ssta_elino", "ssta_lanina", "SOI_index", "NAO_index",
    "tmax_hot_in_planting", "tmax_hot_in_harvesting",
    "tmax_very_hot_in_planting", "tmax_very_hot_in_harvesting",
    "tmin_cold_in_planting", "tmin_cold_in_harvesting",
    "tmin_very_cold_in_planting", "tmin_very_cold_in_harvesting",
    "awnd_moderate_high_wind_in_planting", "awnd_moderate_high_wind_in_harvesting",
    "awnd_extreme_high_wind_in_planting", "awnd_extreme_high_wind_in_harvesting",
    "spi_7d_very_wet_in_planting", "spi_7d_very_wet_in_harvesting",
    "spi_7d_extreme_wet_in_planting", "spi_7d_extreme_wet_in_harvesting",
    "spi_7d_very_dry_in_planting", "spi_7d_very_dry_in_harvesting",
    "spi_7d_extreme_dry_in_planting", "spi_7d_extreme_dry_in_harvesting",
    "pdsi_very_wet_in_planting", "pdsi_very_wet_in_harvesting",
    "pdsi_extreme_wet_in_planting", "pdsi_extreme_wet_in_harvesting",
    "pdsi_extreme_drought_in_planting", "pdsi_extreme_drought_in_harvesting",
    "pdsi_severe_drought_in_planting", "pdsi_severe_drought_in_harvesting",
    "co2_extreme_in_planting", "co2_extreme_in_harvesting",
]
NEWS_COLUMNS  = ["frbsf_sentiment", "Text_Climate_Anomaly", "epu_index"]
MACRO_COLUMNS = ["DJIA_Index", "WTI_Index", "Broad_Dollar_index", "Stock_Uncertainty"]

In [ ]:
# ── Configuration dataclasses (from src/model/rf/types.py) ────────────────────

@dataclass
class WalkForwardConfig:
    window_type: str = "expanding"
    initial_train_size: int = 260
    test_size: int = 4
    step: int = 4
    rolling_window_size: int | None = None
    progress_bar: bool = True


@dataclass
class ModelConfig:
    backend: str = "xgboost_rf"
    device: str = "cuda"
    n_estimators: int = 100
    criterion: str = "squared_error"
    max_depth: int | None = 4
    min_samples_split: int = 4
    min_samples_leaf: int = 2
    max_features: Any = "sqrt"
    bootstrap: bool = True
    random_state: int | None = 42
    n_jobs: int | None = -1
    standardize_features: bool = False
    target_transform: str = "log"
    prediction_floor: float = 1e-10
    log_transform_rv_features: bool = True
    feature_floor: float = 1e-10


@dataclass
class RunConfig:
    walk_forward: WalkForwardConfig = field(default_factory=WalkForwardConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


@dataclass
class BenchmarkConfig:
    csv_path: str = ""
    target_col: str = "wheat_weekly_rv"
    core_columns_by_target: dict | None = None
    target_horizons: list = field(default_factory=lambda: [1, 4, 8, 12, 16])
    target_mode: str = "mean"
    run_configs: dict = field(default_factory=dict)
    climate_columns: list | None = None

In [ ]:
# ── Feature-set utilities (from src/benchmark/utils.py) ───────────────────────

def existing_columns(data: pd.DataFrame, columns: list) -> list:
    return [c for c in columns if c in data.columns]


def infer_target_prefix(target_col: str) -> str:
    if target_col.endswith("_weekly_rv"):
        return target_col[: -len("_weekly_rv")]
    return target_col.split("_", 1)[0]


def default_core_columns(target_col: str) -> list:
    p = infer_target_prefix(target_col)
    return [f"{p}_weekly_rv", f"{p}_monthly_rv", f"{p}_seasonal_rv"]


def default_endo_columns(target_col: str) -> list:
    p = infer_target_prefix(target_col)
    return [f"{p}_weekly_rvb", f"{p}_weekly_rvg", f"{p}_weekly_jumps"]


def default_exo_columns(target_col: str) -> list:
    target_prefix = infer_target_prefix(target_col)
    crop_prefixes = ("wheat", "corn", "soybeans")
    return [f"{cp}_weekly_rv" for cp in crop_prefixes if cp != target_prefix]


def build_feature_sets(data: pd.DataFrame, target_col: str, climate_columns: list | None = None) -> dict:
    endo    = existing_columns(data, default_endo_columns(target_col))
    exo     = existing_columns(data, default_exo_columns(target_col))
    climate = existing_columns(data, climate_columns or CLIMATE_COLUMNS)
    news    = existing_columns(data, NEWS_COLUMNS)
    macro   = existing_columns(data, MACRO_COLUMNS)

    raw = {
        "har":                          [],
        "har_endo":                     endo,
        "har_endo_exo":                 endo + exo,
        "har_endo_exo_news":            endo + exo + news,
        "har_endo_exo_macro":           endo + exo + macro,
        "har_endo_exo_climate":         endo + exo + climate,
        "har_endo_exo_climate_news":    endo + exo + climate + news,
        "har_endo_exo_climate_macro":   endo + exo + climate + macro,
        "har_endo_exo_news_macro":      endo + exo + news + macro,
        "har_endo_exo_climate_news_macro": endo + exo + climate + news + macro,
    }
    cleaned = {}
    for name, cols in raw.items():
        seen: set = set()
        unique = []
        for c in cols:
            if c in data.columns and c not in seen:
                seen.add(c)
                unique.append(c)
        cleaned[name] = unique
    return cleaned


def benchmark_crop_dir_name(target_col: str) -> str:
    prefix = infer_target_prefix(target_col)
    return "soybean" if prefix == "soybeans" else prefix

In [ ]:
# ── Walk-forward window builder (from src/model/common/preprocessing.py) ──────

def build_walk_forward_windows(n_obs: int, cfg: WalkForwardConfig) -> list:
    assert cfg.initial_train_size >= 2
    assert cfg.test_size >= 1
    assert cfg.step >= 1
    assert cfg.initial_train_size + cfg.test_size <= n_obs, \
        f"Not enough observations ({n_obs}) for the walk-forward config."

    windows = []
    test_start = cfg.initial_train_size
    while test_start + cfg.test_size <= n_obs:
        if cfg.window_type == "expanding":
            train_start = 0
        else:
            rolling_size = cfg.rolling_window_size or cfg.initial_train_size
            train_start = max(0, test_start - rolling_size)
        train_end = test_start
        test_end  = test_start + cfg.test_size
        if train_end - train_start >= 2:
            windows.append((train_start, train_end, test_start, test_end))
        test_start += cfg.step

    assert windows, "No walk-forward windows were generated."
    return windows

In [ ]:
# ── Design matrix builder (from src/model/common/preprocessing.py) ────────────

TARGET_COL_NAME = "RV_target"
TARGET_FLOOR    = 1e-10


def build_design_matrix(
    data: pd.DataFrame,
    target_col: str,
    core_cols: list,
    extra_cols: list,
    horizon: int,
    mode: str,
    target_transform: str = "none",
) -> pd.DataFrame:
    feature_cols = core_cols + extra_cols
    design = data[feature_cols].copy()

    base = data[target_col].astype(float)
    if mode == "point":
        design[TARGET_COL_NAME] = base.shift(-horizon)
    else:
        if target_transform == "log":
            base = pd.Series(
                np.log(np.clip(base.to_numpy(dtype=float), TARGET_FLOOR, None)),
                index=data.index,
            )
        design[TARGET_COL_NAME] = (
            base.shift(-1)
            .rolling(window=horizon, min_periods=horizon)
            .mean()
            .shift(-(horizon - 1))
        )
    return design


def split_xy(design: pd.DataFrame) -> tuple:
    clean = design.dropna()
    return clean.drop(columns=[TARGET_COL_NAME]), clean[TARGET_COL_NAME]

In [ ]:
# ── Evaluation metrics (from src/metrics/statistical.py) ──────────────────────

_EPS = 1e-12


def _align(y_true: pd.Series, y_pred: pd.Series) -> tuple:
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    return df["y_true"], df["y_pred"]


def compute_metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
    a, f = _align(y_true, y_pred)
    a_np = a.to_numpy(dtype=float)
    f_np = f.to_numpy(dtype=float)

    mse_val  = float(np.mean((a_np - f_np) ** 2))
    mae_val  = float(np.mean(np.abs(a_np - f_np)))

    ac = np.clip(a_np, _EPS, None)
    fc = np.clip(f_np, _EPS, None)
    qlike_val = float(np.mean(np.log(fc) + ac / fc))

    ss_res = np.sum((a_np - f_np) ** 2)
    ss_tot = np.sum((a_np - a_np.mean()) ** 2)
    r2_val = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

    la = np.log(np.clip(a_np, _EPS, None))
    lf = np.log(np.clip(f_np, _EPS, None))
    ss_res_log = np.sum((la - lf) ** 2)
    ss_tot_log = np.sum((la - la.mean()) ** 2)
    r2log_val  = float(1.0 - ss_res_log / ss_tot_log) if ss_tot_log > 0 else float("nan")

    return {"mse": mse_val, "mae": mae_val, "qlike": qlike_val,
            "r2": r2_val, "r2log": r2log_val, "n_obs": len(a)}

In [ ]:
# ── RF model + feature transforms (from src/model/rf/experiment.py) ───────────

def _resolve_colsample(max_features: Any, n_features: int) -> float:
    if n_features <= 0 or max_features is None:
        return 1.0
    if isinstance(max_features, str):
        if max_features == "sqrt":
            return min(max(math.sqrt(n_features) / n_features, 1.0 / n_features), 1.0)
        if max_features == "log2":
            return min(max(math.log2(max(n_features, 2)) / n_features, 1.0 / n_features), 1.0)
    if isinstance(max_features, int):
        return min(max(max_features / n_features, 1.0 / n_features), 1.0)
    return min(max(float(max_features), 1.0 / n_features), 1.0)


def fit_rf(x_train: pd.DataFrame, y_train: pd.Series, cfg: ModelConfig):
    if cfg.backend == "sklearn":
        if cfg.device != "cpu":
            raise ValueError("sklearn RandomForestRegressor is CPU-only. Use backend='xgboost_rf' for GPU.")
        m = RandomForestRegressor(
            n_estimators=cfg.n_estimators,
            criterion=cfg.criterion,
            max_depth=cfg.max_depth,
            min_samples_split=cfg.min_samples_split,
            min_samples_leaf=cfg.min_samples_leaf,
            max_features=cfg.max_features,
            bootstrap=cfg.bootstrap,
            random_state=cfg.random_state,
            n_jobs=cfg.n_jobs,
        )
    else:
        m = XGBRFRegressor(
            n_estimators=cfg.n_estimators,
            max_depth=6 if cfg.max_depth is None else int(cfg.max_depth),
            subsample=0.8 if cfg.bootstrap else 1.0,
            colsample_bynode=_resolve_colsample(cfg.max_features, x_train.shape[1]),
            min_child_weight=float(cfg.min_samples_leaf),
            reg_lambda=0.0,
            learning_rate=1.0,
            objective="reg:squarederror",
            random_state=cfg.random_state,
            n_jobs=cfg.n_jobs,
            tree_method="hist",
            device=cfg.device,
        )
    m.fit(x_train, y_train)
    return m


def log_transform_rv_features(x: pd.DataFrame, floor: float) -> tuple:
    x = x.copy()
    transformed = []
    for col in x.columns:
        name = col.lower()
        if name == "rv" or name.endswith("_rv"):
            x[col] = np.log(np.clip(x[col].to_numpy(dtype=float), floor, None))
            transformed.append(col)
    return x, transformed


def transform_target(y: pd.Series, cfg: ModelConfig) -> pd.Series:
    if cfg.target_transform == "none":
        return y
    return pd.Series(
        np.log(np.clip(y.to_numpy(dtype=float), cfg.prediction_floor, None)),
        index=y.index, name=y.name,
    )


def inverse_transform(y: pd.Series, cfg: ModelConfig) -> pd.Series:
    if cfg.target_transform == "none":
        arr = np.clip(y.to_numpy(dtype=float), cfg.prediction_floor, None)
    else:
        arr = np.clip(np.exp(y.to_numpy(dtype=float)), cfg.prediction_floor, None)
    return pd.Series(arr, index=y.index, name=y.name)


def standardize_train_test(x_train: pd.DataFrame, x_test: pd.DataFrame) -> tuple:
    scaler = StandardScaler()
    scaler.fit(x_train)
    xt  = pd.DataFrame(scaler.transform(x_train), index=x_train.index, columns=x_train.columns)
    xte = pd.DataFrame(scaler.transform(x_test),  index=x_test.index,  columns=x_test.columns)
    return xt, xte


def aggregate_predictions(true_parts: list, pred_parts: list) -> tuple:
    y_true = pd.concat(true_parts).groupby(level=0).mean().sort_index()
    y_pred = pd.concat(pred_parts).groupby(level=0).mean().sort_index()
    df = y_true.to_frame("y_true").join(y_pred.to_frame("y_pred"), how="inner").dropna()
    return df["y_true"], df["y_pred"]

In [ ]:
# ── Walk-forward experiment loop (from src/model/rf/experiment.py) ─────────────

def run_experiment(x: pd.DataFrame, y: pd.Series, run_cfg: RunConfig) -> dict:
    wf_cfg    = run_cfg.walk_forward
    model_cfg = run_cfg.model

    if model_cfg.log_transform_rv_features:
        x, _ = log_transform_rv_features(x, floor=model_cfg.feature_floor)

    windows = build_walk_forward_windows(len(x), wf_cfg)
    train_true_parts, train_pred_parts = [], []
    test_true_parts,  test_pred_parts  = [], []
    last_importances = pd.Series(dtype=float)

    for train_start, train_end, test_start, test_end in tqdm(
        windows,
        desc=f"RF walk-forward ({wf_cfg.window_type})",
        disable=not wf_cfg.progress_bar,
    ):
        x_train = x.iloc[train_start:train_end]
        y_train = y.iloc[train_start:train_end]
        x_test  = x.iloc[test_start:test_end]
        y_test  = y.iloc[test_start:test_end]

        if model_cfg.standardize_features:
            x_train, x_test = standardize_train_test(x_train, x_test)

        y_train_model = transform_target(y_train, model_cfg)
        fitted = fit_rf(x_train, y_train_model, model_cfg)

        y_pred_train = inverse_transform(
            pd.Series(fitted.predict(x_train), index=x_train.index), model_cfg
        )
        y_pred_test = inverse_transform(
            pd.Series(fitted.predict(x_test), index=x_test.index), model_cfg
        )

        train_true_parts.append(y_train)
        train_pred_parts.append(y_pred_train)
        test_true_parts.append(y_test)
        test_pred_parts.append(y_pred_test)

        last_importances = pd.Series(
            fitted.feature_importances_, index=x.columns, name="importance"
        ).sort_values(ascending=False)

    y_true_train, y_pred_train_all = aggregate_predictions(train_true_parts, train_pred_parts)
    y_true_test,  y_pred_test_all  = aggregate_predictions(test_true_parts,  test_pred_parts)

    return {
        "y_true_train":      y_true_train,
        "y_pred_train":      y_pred_train_all,
        "y_true_test":       y_true_test,
        "y_pred_test":       y_pred_test_all,
        "metrics": {
            "train": compute_metrics(y_true_train, y_pred_train_all),
            "test":  compute_metrics(y_true_test,  y_pred_test_all),
        },
        "feature_importances": last_importances,
        "selected_features":   x.columns.tolist(),
        "n_windows":           len(windows),
        "window_type":         wf_cfg.window_type,
        "model_info": {
            "window_type":                     wf_cfg.window_type,
            "initial_train_size":              wf_cfg.initial_train_size,
            "test_size":                       wf_cfg.test_size,
            "step":                            wf_cfg.step,
            "rolling_window_size":             wf_cfg.rolling_window_size,
            "n_windows":                       len(windows),
            "rf_backend":                      model_cfg.backend,
            "rf_device":                       model_cfg.device,
            "rf_n_estimators":                 model_cfg.n_estimators,
            "rf_max_depth":                    model_cfg.max_depth,
            "rf_min_samples_split":            model_cfg.min_samples_split,
            "rf_min_samples_leaf":             model_cfg.min_samples_leaf,
            "rf_max_features":                 model_cfg.max_features,
            "rf_bootstrap":                    model_cfg.bootstrap,
            "rf_random_state":                 model_cfg.random_state,
            "rf_n_jobs":                       model_cfg.n_jobs,
            "model_target_transform":          model_cfg.target_transform,
            "model_log_transform_rv_features": model_cfg.log_transform_rv_features,
        },
    }

In [ ]:
# ── Single feature-set / horizon runner (from src/model/rf/experiment.py) ──────

def run_feature_horizon(
    data: pd.DataFrame,
    target_col: str,
    core_cols: list,
    extra_cols: list,
    horizon: int,
    target_mode: str,
    run_cfg: RunConfig,
) -> dict:
    model_cfg = run_cfg.model

    # When mode=mean and transform=log, the design matrix already applies log
    # to the target, so the model must NOT apply a second log transform.
    mean_log = (target_mode == "mean" and model_cfg.target_transform == "log")
    design_transform = model_cfg.target_transform  # used by the design matrix

    design = build_design_matrix(
        data, target_col, core_cols, extra_cols,
        horizon, target_mode, target_transform=design_transform,
    )
    x, y = split_xy(design)

    effective_cfg = run_cfg
    if mean_log:
        # Disable double-transform: model receives already-log targets
        eff_model = dataclasses.replace(
            model_cfg, target_transform="none", prediction_floor=-1e12
        )
        effective_cfg = dataclasses.replace(run_cfg, model=eff_model)

    result = run_experiment(x, y, effective_cfg)

    if mean_log:
        # Inverse-transform back to original scale for metrics
        inv_cfg = dataclasses.replace(model_cfg, target_transform="log", prediction_floor=1e-10)
        for key in ("y_true_train", "y_pred_train", "y_true_test", "y_pred_test"):
            result[key] = inverse_transform(result[key], inv_cfg)
        result["metrics"] = {
            "train": compute_metrics(result["y_true_train"], result["y_pred_train"]),
            "test":  compute_metrics(result["y_true_test"],  result["y_pred_test"]),
        }

    result.update({
        "target_col":        target_col,
        "target_horizon":    horizon,
        "target_mode":       target_mode,
        "extra_feature_cols": extra_cols,
    })
    result["model_info"].update({
        "target_col_raw":     target_col,
        "target_horizon":     horizon,
        "target_mode":        target_mode,
        "extra_feature_cols": extra_cols,
    })
    return result

In [ ]:
# ── Multi-horizon benchmark runner (from src/benchmark/rf/runner.py) ───────────

def run_benchmark(cfg: BenchmarkConfig, data: pd.DataFrame | None = None) -> dict:
    """
    Returns: {horizon: {run_name: {feature_set_name: result_dict}}}
    """
    if data is None:
        csv_path = cfg.csv_path
        # Resolve relative paths against DATA_DIR
        p = Path(csv_path)
        if not p.is_absolute():
            parts = p.parts
            skip = 1 if parts[0] == "data" else 0
            csv_path = str(DATA_DIR / Path(*parts[skip:]))
        data = pd.read_csv(csv_path)
        logging.info("Loaded %d rows from %s", len(data), csv_path)

    if cfg.core_columns_by_target and cfg.target_col in cfg.core_columns_by_target:
        core_cols = cfg.core_columns_by_target[cfg.target_col]
    else:
        core_cols = existing_columns(data, default_core_columns(cfg.target_col))

    feature_sets = build_feature_sets(data, cfg.target_col, cfg.climate_columns)
    results = {h: {rn: {} for rn in cfg.run_configs} for h in cfg.target_horizons}

    total = len(cfg.target_horizons) * len(cfg.run_configs) * len(feature_sets)
    done  = 0
    for horizon in cfg.target_horizons:
        for run_name, run_cfg in cfg.run_configs.items():
            for fs_name, extra_cols in feature_sets.items():
                done += 1
                t0 = time.perf_counter()
                logging.info(
                    "[%d/%d] horizon=%d  run=%s  feature_set=%s",
                    done, total, horizon, run_name, fs_name,
                )
                result = run_feature_horizon(
                    data, cfg.target_col, core_cols, extra_cols,
                    horizon, cfg.target_mode, run_cfg,
                )
                results[horizon][run_name][fs_name] = result
                logging.info(
                    "  done in %.1fs  test_r2=%.4f  test_mse=%.3e",
                    time.perf_counter() - t0,
                    result["metrics"]["test"]["r2"],
                    result["metrics"]["test"]["mse"],
                )
    return results

In [ ]:
# ── JSON config loader + results + checkpoint helpers ─────────────────────────
# (from scripts/benchmark/random_forest.py and src/benchmark/checkpoints.py)

def load_config(config_path: Path) -> BenchmarkConfig:
    with config_path.open(encoding="utf-8") as f:
        raw = json.load(f)

    run_configs  = {}
    wf_fields    = set(WalkForwardConfig.__dataclass_fields__)
    model_fields = set(ModelConfig.__dataclass_fields__)

    for name, rc in (raw.get("run_configs") or {}).items():
        wf_raw = rc.get("walk_forward") or {}
        m_raw  = rc.get("model") or {}
        run_configs[name] = RunConfig(
            walk_forward=WalkForwardConfig(**{k: v for k, v in wf_raw.items() if k in wf_fields}),
            model=ModelConfig(**{k: v for k, v in m_raw.items() if k in model_fields}),
        )

    return BenchmarkConfig(
        csv_path=raw.get("csv_path", ""),
        target_col=raw.get("target_col", "wheat_weekly_rv"),
        core_columns_by_target=raw.get("core_columns_by_target"),
        target_horizons=[int(h) for h in raw.get("target_horizons", [1])],
        target_mode=raw.get("target_mode", "mean"),
        run_configs=run_configs,
        climate_columns=raw.get("climate_columns"),
    )


def results_to_frame(results: dict) -> pd.DataFrame:
    rows = []
    for horizon in sorted(results):
        for run_name, feature_results in results[horizon].items():
            for fs_name, r in feature_results.items():
                rows.append({
                    "target_horizon": horizon,
                    "model_type":     run_name,
                    "feature_set":    fs_name,
                    "window_type":    r.get("window_type"),
                    "target_mode":    r.get("target_mode"),
                    "train_r2":    r["metrics"]["train"]["r2"],
                    "test_r2":     r["metrics"]["test"]["r2"],
                    "train_mse":   r["metrics"]["train"]["mse"],
                    "test_mse":    r["metrics"]["test"]["mse"],
                    "train_mae":   r["metrics"]["train"]["mae"],
                    "test_mae":    r["metrics"]["test"]["mae"],
                    "train_qlike": r["metrics"]["train"]["qlike"],
                    "test_qlike":  r["metrics"]["test"]["qlike"],
                    "train_r2log": r["metrics"]["train"]["r2log"],
                    "test_r2log":  r["metrics"]["test"]["r2log"],
                    "n_windows":   r.get("n_windows"),
                })
    return pd.DataFrame(rows).sort_values(
        ["target_horizon", "test_r2"], ascending=[True, False]
    ).reset_index(drop=True)


# ── Checkpoint saving (from src/benchmark/checkpoints.py) ─────────────────────

def _safe_name(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip())
    return cleaned.strip("_") or "unknown"


def _jsonable(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonable(v) for v in value]
    if isinstance(value, pd.Series):
        return {str(k): _jsonable(v) for k, v in value.to_dict().items()}
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="records")
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def save_checkpoints(results: dict, checkpoint_root: Path, target_mode: str) -> list:
    """Save per-run checkpoint bundles, mirroring save_best_result_checkpoints."""
    saved = []
    for horizon in sorted(results):
        for model_type, fs_results in sorted(results[horizon].items()):
            for feature_set, r in sorted(fs_results.items()):
                ckpt_dir = (
                    checkpoint_root
                    / f"target_horizon_{horizon}"
                    / "checkpoints"
                    / f"{_safe_name(model_type)}__{_safe_name(feature_set)}"
                )
                ckpt_dir.mkdir(parents=True, exist_ok=True)

                metadata = {
                    "model_family":    "rf",
                    "target_mode":     target_mode,
                    "target_horizon":  horizon,
                    "model_type":      model_type,
                    "feature_set":     feature_set,
                    "saved_at_utc":    datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    "checkpoint_dir":  str(ckpt_dir),
                }
                (ckpt_dir / "metadata.json").write_text(
                    json.dumps(metadata, indent=2), encoding="utf-8"
                )
                (ckpt_dir / "metrics.json").write_text(
                    json.dumps(_jsonable(r.get("metrics", {})), indent=2), encoding="utf-8"
                )

                model_info = dict(r.get("model_info", {}))
                (ckpt_dir / "model_info.json").write_text(
                    json.dumps(_jsonable(model_info), indent=2), encoding="utf-8"
                )

                selected = r.get("selected_features") or []
                (ckpt_dir / "selected_features.txt").write_text(
                    "\n".join(str(c) for c in selected), encoding="utf-8"
                )

                for split in ("train", "test"):
                    y_true = r.get(f"y_true_{split}")
                    y_pred = r.get(f"y_pred_{split}")
                    if y_true is not None and y_pred is not None:
                        pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).to_csv(
                            ckpt_dir / f"{split}_predictions.csv"
                        )

                fi = r.get("feature_importances")
                if fi is not None and len(fi) > 0:
                    fi.to_frame("importance").to_csv(ckpt_dir / "feature_importances.csv")

                saved.append(ckpt_dir)
    return saved


def save_results(results: dict, cfg: BenchmarkConfig, output_dir: Path) -> None:
    summary = results_to_frame(results)
    output_dir.mkdir(parents=True, exist_ok=True)

    for horizon in sorted(results):
        h_dir = output_dir / f"target_horizon_{horizon}"
        h_dir.mkdir(parents=True, exist_ok=True)
        h_df = summary[summary["target_horizon"] == horizon].copy()
        h_df.to_csv(h_dir / "random_forest.csv", index=False)
        logging.info("Saved horizon=%d summary → %s", horizon, h_dir / "random_forest.csv")

    saved = save_checkpoints(results, checkpoint_root=output_dir, target_mode=cfg.target_mode)
    logging.info("Saved %d checkpoint bundles → %s", len(saved), output_dir)


def run_crop(config_path: Path, crop_name: str) -> pd.DataFrame:
    logging.info("=== %s ===", crop_name.upper())
    cfg = load_config(config_path)
    output_dir = DATA_DIR / "benchmark" / benchmark_crop_dir_name(cfg.target_col) / "rf" / cfg.target_mode
    results = run_benchmark(cfg)
    save_results(results, cfg, output_dir)
    logging.info("=== %s DONE ===", crop_name.upper())
    return results_to_frame(results)

## Run Benchmarks

Each cell runs all horizons (1, 4, 8, 12, 16 weeks) × all feature sets × both run configs for one crop.
Results are saved to `DATA_DIR/benchmark/{crop}/rf/mean/`.

In [ ]:
wheat_summary = run_crop(CONFIG_WHEAT, "wheat")
wheat_summary[["target_horizon", "model_type", "feature_set", "test_r2", "test_mse"]].head(10)

In [ ]:
corn_summary = run_crop(CONFIG_CORN, "corn")
corn_summary[["target_horizon", "model_type", "feature_set", "test_r2", "test_mse"]].head(10)

In [ ]:
soybean_summary = run_crop(CONFIG_SOYBEAN, "soybean")
soybean_summary[["target_horizon", "model_type", "feature_set", "test_r2", "test_mse"]].head(10)

In [ ]:
all_summary = pd.concat(
    [
        wheat_summary.assign(crop="wheat"),
        corn_summary.assign(crop="corn"),
        soybean_summary.assign(crop="soybean"),
    ],
    ignore_index=True,
)

best = (
    all_summary
    .sort_values("test_r2", ascending=False)
    .groupby(["crop", "target_horizon"], as_index=False)
    .first()[["crop", "target_horizon", "model_type", "feature_set", "test_r2", "test_mse"]]
    .sort_values(["crop", "target_horizon"])
    .reset_index(drop=True)
)
print("Best model per crop × horizon:")
print(best.to_string(index=False))